<a href="https://colab.research.google.com/github/jaswanth-098/Secure-login-system/blob/main/Secure_login_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
import secrets
import hashlib
import hmac
import random
import gradio as gr

DATABASE = "/content/users.db"
ITERATIONS = 600_000

def get_db():
    return sqlite3.connect(DATABASE)


with get_db() as db:
    db.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            salt BLOB NOT NULL,
            password_hash BLOB NOT NULL
        )
    """)

def hash_password(password, salt):
    return hashlib.pbkdf2_hmac(
        "sha256", password.encode("utf-8"),
        salt, ITERATIONS
    )

def register(username, password):
    username = username.strip()

    if not username or not password:
        return "Enter username and password."

    if len(username) > 80 or len(password) > 1024:
        return "Input is too long."

    salt = secrets.token_bytes(16)
    password_hash = hash_password(password, salt)

    try:
        with get_db() as db:
            db.execute("""
                INSERT INTO users (username, salt, password_hash)
                VALUES (?, ?, ?)
            """, (username, salt, password_hash))

        return "Registration successful! You can now log in."

    except sqlite3.IntegrityError:
        return "Username already exists."

def new_captcha():
    a = random.randint(1, 9)
    b = random.randint(1, 9)
    return f"What is {a} + {b}?", str(a + b)

def login(username, password, captcha_input, expected_answer):
    username = (username or "").strip()

    # Check CAPTCHA first
    if not (captcha_input or "").strip().isdigit():
        return "Please enter the correct CAPTCHA answer."

    if not hmac.compare_digest(
        (captcha_input or "").strip(),
        expected_answer or ""
    ):
        return "Incorrect CAPTCHA. Refresh and try again."

    # Verify password
    with get_db() as db:
        user = db.execute("""
            SELECT salt, password_hash
            FROM users
            WHERE username = ?
        """, (username,)).fetchone()

    if user is None:
        return "Login failed: invalid username or password."

    salt, stored_hash = user
    entered_hash = hash_password(password or "", salt)

    if not hmac.compare_digest(entered_hash, stored_hash):
        return "Login failed: invalid username or password."

    return f"Login successful! Welcome, {username}."

def refresh_captcha():
    question, answer = new_captcha()
    return question, answer, ""

initial_question, initial_answer = new_captcha()

with gr.Blocks(title="Secure Login System") as app:
    gr.Markdown("""
    # 🔐 Secure Login System
    ### Password Hashing + CAPTCHA Verification
    """)

    with gr.Tab("Register"):
        reg_username = gr.Textbox(label="Create username")
        reg_password = gr.Textbox(
            label="Create password", type="password"
        )
        reg_button = gr.Button("Register")
        reg_result = gr.Textbox(label="Registration result")

        reg_button.click(
            register,
            inputs=[reg_username, reg_password],
            outputs=reg_result
        )

    with gr.Tab("Login"):
        username = gr.Textbox(label="Username")
        password = gr.Textbox(
            label="Password", type="password"
        )

        question = gr.Textbox(
            label="CAPTCHA Challenge",
            value=initial_question,
            interactive=False
        )


        answer_state = gr.State(initial_answer)

        captcha_input = gr.Textbox(
            label="Enter CAPTCHA answer",
            placeholder="Enter the answer"
        )

        with gr.Row():
            login_button = gr.Button("Login securely")
            refresh_button = gr.Button("New CAPTCHA")

        login_result = gr.Textbox(
            label="Login result", interactive=False
        )

        login_button.click(
            login,
            inputs=[
                username, password, captcha_input, answer_state
            ],
            outputs=login_result
        )

        refresh_button.click(
            refresh_captcha,
            outputs=[question, answer_state, captcha_input]
        )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6e9a63243a47116ede.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
